In [1]:
pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.9.1
Uninstalling torch-2.9.1:
  Successfully uninstalled torch-2.9.1
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu128

Looking in indexes: https://download.pytorch.org/whl/nightly/cu128
   ---------------------------------------- 0.0/2.9 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.9 GB 5.8 MB/s eta 0:08:11
   ---------------------------------------- 0.0/2.9 GB 4.3 MB/s eta 0:11:07
   ---------------------------------------- 0.0/2.9 GB 4.0 MB/s eta 0:11:52
   ---------------------------------------- 0.0/2.9 GB 4.1 MB/s eta 0:11:47
   ---------------------------------------- 0.0/2.9 GB 4.0 MB/s eta 0:11:53
   ---------------------------------------- 0.0/2.9 GB 4.0 MB/s eta 0:11:56
   ---------------------------------------- 0.0/2.9 GB 4.0 MB/s eta 0:12:00
   ---------------------------------------- 0.0/2.9 GB 4.0 MB/s eta 0:12:02
   ---------------------------------------- 0.0/2.9 GB 4.0 MB/s eta 0:12:04
   ---------------------------------------- 0.0/2.9 GB 3.9 MB/s eta 0:12:05
   ---------------------------------------- 0.0/2.9 GB 3.9 MB/s eta 0:12:07
   -------------------------

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

# THE CRITICAL TEST - this is where cu124 failed before
print("\nTesting GPU computation...")
try:
    x = torch.randn(1000, 1000).cuda()
    y = torch.randn(1000, 1000).cuda()
    z = torch.matmul(x, y)
    print("✓✓✓ SUCCESS! GPU computation works!")
    print("✓✓✓ You can train on GPU now!")
except RuntimeError as e:
    print("✗✗✗ FAILED:", e)
    print("sm_120 still not supported in cu128")
    print("Use CPU training instead")

PyTorch version: 2.11.0.dev20260113+cu128
CUDA version: 12.8
CUDA available: True
GPU name: NVIDIA GeForce RTX 5060 Laptop GPU

Testing GPU computation...
✓✓✓ SUCCESS! GPU computation works!
✓✓✓ You can train on GPU now!


In [9]:
pip install datasets transformers

  Using cached datasets-4.4.2-py3-none-any.whl.metadata (19 kB)
  Using cached pyarrow-22.0.0-cp313-cp313-win_amd64.whl.metadata (3.3 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached xxhash-3.6.0-cp313-cp313-win_amd64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.18-py313-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
  Using cached huggingface_hub-1.3.1-py3-none-any.whl.metadata (13 kB)
  Using cached aiohttp-3.13.3-cp313-cp313-win_amd64.whl.metadata (8.4 kB)
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached hf_xet-1.2.0-cp37-abi3-win_amd64.whl.metadata (5.0 kB)
  Using cached typer_slim-0.21.1-py3-none-any.whl.metadata (16 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.meta

In [2]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score
import random

c:\Users\Chinmay\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
from sklearn.metrics import confusion_matrix, classification_report

In [3]:
CSV_PATH = "converted_messages.csv"
TEXT_COL = "Message"
LABEL_COL = "Label"

In [4]:
MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = "./sms_model"

In [5]:
SAMPLE_SIZE = 300_000
MAX_LENGTH = 128
EPOCHS = 2
BATCH_SIZE = 16
LR = 2e-5
SEED = 42

In [6]:
LABEL2ID = {
    "NotTransaction": 0,
    "Transaction": 1,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

In [7]:
## SEED

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [8]:
## Loading data

print("Loading dataset...")
df = pd.read_csv(CSV_PATH, usecols=[TEXT_COL, LABEL_COL])
df = df[df[LABEL_COL].isin(LABEL2ID)]
df = df.dropna()

Loading dataset...


In [9]:
if len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=SEED)

df["label"] = df[LABEL_COL].map(LABEL2ID)

In [10]:
df["label"] = df[LABEL_COL].map(LABEL2ID)
print(df.columns)
print(df.head())

Index(['Message', 'Label', 'label'], dtype='object')
                                                  Message           Label  \
17220   Don't tell anyone the code 814461!  Varenichna...  NotTransaction   
334620  Transaction Error: We encountered an issue pro...  NotTransaction   
526629  Payment Alert: 250.87 INR at RELO Direct 5. Ca...     Transaction   
945555  Credit of 3734.92 INR from UniCredit to accoun...     Transaction   
65925   Do not tell anyone the code: 117253. After con...  NotTransaction   

        label  
17220       0  
334620      0  
526629      1  
945555      1  
65925       0  


In [11]:
print("Dataset size:", len(df))
print(df.head())

Dataset size: 300000
                                                  Message           Label  \
17220   Don't tell anyone the code 814461!  Varenichna...  NotTransaction   
334620  Transaction Error: We encountered an issue pro...  NotTransaction   
526629  Payment Alert: 250.87 INR at RELO Direct 5. Ca...     Transaction   
945555  Credit of 3734.92 INR from UniCredit to accoun...     Transaction   
65925   Do not tell anyone the code: 117253. After con...  NotTransaction   

        label  
17220       0  
334620      0  
526629      1  
945555      1  
65925       0  


In [12]:
dataset = Dataset.from_pandas(
    df[[TEXT_COL, "label"]],
    preserve_index=False
)

In [13]:
print("Dataset columns:", dataset.column_names)

Dataset columns: ['Message', 'label']


In [14]:
## Train test

dataset = dataset.train_test_split(test_size=0.1, seed=SEED)
train_ds = dataset["train"]
val_ds = dataset["test"]

In [15]:
## Tokenize

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch[TEXT_COL],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)

Map: 100%|██████████| 30000/30000 [00:01<00:00, 22936.05 examples/s]


In [16]:
print("Train columns:", train_ds.column_names)
print("Val columns:", val_ds.column_names)

Train columns: ['Message', 'label', 'input_ids', 'attention_mask']
Val columns: ['Message', 'label', 'input_ids', 'attention_mask']


In [17]:
## Model

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
## Metrics

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds)
    }

In [28]:
pip install transformers[torch]

  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.12.0-py3-none-any.whl (380 kB)
Note: you may need to restart the kernel to use updated packages.


In [19]:
## Training arguments

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=200,
    seed=SEED,
    fp16=torch.cuda.is_available(),
)


In [20]:
## Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

C:\Users\Chinmay\AppData\Local\Temp\ipykernel_11504\3523573270.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [26]:
import importlib.util
print(importlib.util.find_spec("torch"))

ModuleSpec(name='torch', loader=<_frozen_importlib_external.SourceFileLoader object at 0x000001C144DA8C90>, origin='c:\\Users\\Chinmay\\Desktop\\Proj\\ExpenseEase-v2\\.venv311\\Lib\\site-packages\\torch\\__init__.py', submodule_search_locations=['c:\\Users\\Chinmay\\Desktop\\Proj\\ExpenseEase-v2\\.venv311\\Lib\\site-packages\\torch'])


In [21]:
print("\nStarting training...\n")
trainer.train()


Starting training...



Epoch,Training Loss,Validation Loss,Accuracy
1,0.000000,0.000424,0.999967
2,0.000000,0.000524,0.999967


TrainOutput(global_step=33750, training_loss=0.0007803933790436498, metrics={'train_runtime': 1746.0175, 'train_samples_per_second': 309.275, 'train_steps_per_second': 19.33, 'total_flos': 1.788309881856e+16, 'train_loss': 0.0007803933790436498, 'epoch': 2.0})

In [24]:
pred_output = trainer.predict(val_ds)

y_true = pred_output.label_ids
y_pred = pred_output.predictions.argmax(axis=1)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Non-Transactional", "Transactional"]
))



Confusion Matrix:
[[12815     0]
 [    1 17184]]

Classification Report:
                   precision    recall  f1-score   support

Non-Transactional       1.00      1.00      1.00     12815
    Transactional       1.00      1.00      1.00     17185

         accuracy                           1.00     30000
        macro avg       1.00      1.00      1.00     30000
     weighted avg       1.00      1.00      1.00     30000



In [25]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved to:", OUTPUT_DIR)

Model saved to: ./sms_model
